# **Perturbation: ablation, occlusion and RISE on a single picture**

Practice for the module [«Explaining DL models through perturbation»](https://ai-interpretability.school).

All three methods answer the same question — «what happens to the prediction if part of the
input is removed» — and differ only in **what exactly** they remove and **how** they gather the
answers into a map. Here we build them ourselves, on one photograph and one model.

And then we do the thing this practice was written for: **we compare them by a number, not by
eye**. And discover that the faithfulness of a map depends on more than the map: change one
word in the metric and random noise takes the lead.

In [ ]:
import torch
import numpy as np
import requests
import matplotlib.pyplot as plt
from io import BytesIO
from PIL import Image, ImageFilter
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights

torch.manual_seed(0)
np.random.seed(0)

model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model.eval();

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

RAW = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/'

def load(name):
    return Image.open(BytesIO(requests.get(RAW + name).content)).convert('RGB')

image = load('pig.png').resize((224, 224))
x = transform(image).unsqueeze(0)

categories = [s.strip() for s in requests.get(RAW + 'imagenet_classes.txt').text.splitlines()]
with torch.no_grad():
    probs = model(x).softmax(1)
target = probs.argmax().item()
print(f'Класс: {target} — {categories[target]}, уверенность {probs[0, target]:.3f}')

## 1. The baseline: what we replace the removed part with

All three methods share a detail that the lesson mentions in one line and that decides
everything in practice: **what the removed part of the input is replaced with**. With zero?
Which zero — a black pixel, or zero in normalized coordinates?

It is the same question as the baseline of Integrated Gradients, and the same trap:
`torch.zeros_like(x)` after normalization is **not black** but grey. Let us check.

In [ ]:
black = transform(Image.new('RGB', (224, 224), (0, 0, 0)))
blur = transform(image.filter(ImageFilter.GaussianBlur(12)))

print(f'zero tensor      → mean over channels: {(x * 0).mean():.4f}')
print(f'true black       → same coordinates:    {black.mean():.4f}')

def predict(batch):
    with torch.no_grad():
        return model(batch).softmax(1)[:, target]

print(f'\nconfidence on the zero tensor: {predict(torch.zeros_like(x)).item():.4f}')
print(f'confidence on a black frame:     {predict(black.unsqueeze(0)).item():.4f}')
print(f'confidence on the blur:          {predict(blur.unsqueeze(0)).item():.4f}')

# Blur as a baseline: it removes detail without creating in its place an object
# that does not occur in nature. A black rectangle in a photograph does create one.
BASELINE = blur
base_score = predict(x).item()

## 2. Feature Ablation

The formula from the lesson: $AblImp_S(x) = f(x) - f(x^{(S\to b)})$. For an image a «feature» is
a pixel, and the first temptation is to compute it for each one. Let us do that.

In [ ]:
# Ablation over single pixels: on a coarse grid, otherwise it is 50 176 passes.
step = 16
coords = [(i, j) for i in range(0, 224, step) for j in range(0, 224, step)]
batch = x.repeat(len(coords), 1, 1, 1)
for k, (i, j) in enumerate(coords):
    batch[k, :, i, j] = BASELINE[:, i, j]

scores = torch.cat([predict(batch[s:s + 64]) for s in range(0, len(batch), 64)])
single = base_score - scores
print(f'model confidence:        {base_score:.4f}')
print(f'maximum over a single pixel: {single.max():.6f} — {single.max() / base_score:.2%} of the confidence')
print(f'mean over a single pixel:     {single.mean():.6f} — {single.mean() / base_score:.3%}')

The mean contribution of a single pixel is a thousandth of a percent, and with a minus
sign at that: removing a random pixel on average slightly *raises* the confidence. The maximum
is a fifth of a percent. This is not «pixels do not matter», this is **saturation**: the network
restores one lost pixel from its neighbours without effort, and the difference of two nearly
equal numbers drowns in noise.

Hence the practical rule of the lesson: for images, perturbation is always **grouped**.
Let us repeat it with $16\times16$ blocks.

In [ ]:
def ablate_regions(size):
    """Ablation with disjoint size x size blocks — one pass per block."""
    cells = [(i, j) for i in range(0, 224, size) for j in range(0, 224, size)]
    batch = x.repeat(len(cells), 1, 1, 1)
    for k, (i, j) in enumerate(cells):
        batch[k, :, i:i + size, j:j + size] = BASELINE[:, i:i + size, j:j + size]
    scores = torch.cat([predict(batch[s:s + 32]) for s in range(0, len(batch), 32)])
    heat = torch.zeros(224, 224)
    for (i, j), s in zip(cells, scores):
        heat[i:i + size, j:j + size] = base_score - s
    return heat

abl = ablate_regions(16)
print(f'ablation of a 16x16 block: maximum {abl.max():.4f} — {abl.max() / base_score:.1%} of the confidence')

## 3. Occlusion

There is exactly one difference from ablation: the window **slides with an overlap**, and the
contribution of a pixel is averaged over all the windows that covered it — the formula of the
lesson $Occ_i(x)=\frac1k\sum_j\big(f(x)-f(x^{(R_j\to b)})\big)$.

In [ ]:
def occlusion(size, stride):
    """A sliding window with overlap; a pixel gets the mean over the windows covering it."""
    cells = [(i, j) for i in range(0, 224 - size + 1, stride)
                    for j in range(0, 224 - size + 1, stride)]
    batch = x.repeat(len(cells), 1, 1, 1)
    for k, (i, j) in enumerate(cells):
        batch[k, :, i:i + size, j:j + size] = BASELINE[:, i:i + size, j:j + size]
    scores = torch.cat([predict(batch[s:s + 32]) for s in range(0, len(batch), 32)])

    total, count = torch.zeros(224, 224), torch.zeros(224, 224)
    for (i, j), s in zip(cells, scores):
        total[i:i + size, j:j + size] += base_score - s
        count[i:i + size, j:j + size] += 1
    return total / count, len(cells)

occ_64, n64 = occlusion(64, 16)
occ_32, n32 = occlusion(32, 16)
print(f'window 64, stride 16: {n64} forward passes')
print(f'window 32, stride 16: {n32} forward passes')

## 4. RISE

Three steps from the lesson: a small Bernoulli grid $h\times w$, an upsampling by bilinear
interpolation with a random shift, a weighted average of the masks. The normalization is the
familiar $\frac{1}{N\cdot\mathbb{E}[M]}$, where $\mathbb{E}[M]=p$.

In [ ]:
def rise(n_masks, p=0.5, s=7, batch_size=50):
    """RISE: an s x s Bernoulli grid → upsampling → random shift → crop to 224."""
    cell = int(np.ceil(224 / s))
    up = (s + 1) * cell
    heat = torch.zeros(224, 224)
    done = 0
    while done < n_masks:
        k = min(batch_size, n_masks - done)
        grid = (torch.rand(k, 1, s, s) < p).float()
        big = torch.nn.functional.interpolate(grid, size=(up, up),
                                              mode='bilinear', align_corners=False)
        masks = torch.empty(k, 1, 224, 224)
        for m in range(k):
            di, dj = np.random.randint(0, cell, 2)
            masks[m] = big[m, :, di:di + 224, dj:dj + 224]
        scores = predict(x * masks)
        heat += (masks[:, 0] * scores[:, None, None]).sum(0)
        done += k
    return heat / (n_masks * p)

rise_50 = rise(50)
rise_200 = rise(200)
rise_500 = rise(500)
print('RISE computed for 50, 200 and 500 masks')

In [ ]:
def show(ax, heat, title):
    ax.imshow(image)
    ax.imshow(heat.numpy(), cmap='jet', alpha=0.5)
    ax.set_title(title, fontsize=10)
    ax.axis('off')

MAPS = {'Ablation, blocks 16': abl, 'Occlusion, window 64': occ_64, 'Occlusion, window 32': occ_32,
        'RISE, 50 masks': rise_50, 'RISE, 200 masks': rise_200, 'RISE, 500 masks': rise_500}

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (name, heat) in zip(axes.flat, MAPS.items()):
    show(ax, heat, name)
plt.tight_layout()
plt.show()

## 5. Which of them is right

By eye the maps differ, and one of them surely looks more convincing. But «more convincing» is
not «more faithful»: that is what the whole block on evaluating explanations is about. Let us
measure, with two standard metrics.

**Deletion.** We remove pixels in decreasing order of importance and watch the confidence fall.
The **lower** the area under the curve, the better.

**Insertion.** The other way round: we start from the blurred picture and put pixels back in
decreasing order of importance. The **higher** the area, the better.

We add a random map to the comparison — it should come out worst on both metrics.

In [ ]:
def curve_auc(heat, mode, steps=100):
    """deletion: remove from the picture; insertion: put back into the blur."""
    order = heat.flatten().argsort(descending=True)
    per = len(order) // steps
    cur = (x if mode == 'deletion' else BASELINE.unsqueeze(0)).clone()
    src = BASELINE if mode == 'deletion' else x[0]
    curve = [predict(cur).item() / base_score]
    for t in range(steps):
        idx = order[t * per:(t + 1) * per]
        rows, cols = idx // 224, idx % 224
        cur[0, :, rows, cols] = src[:, rows, cols]
        curve.append(predict(cur).item() / base_score)
    return float(np.trapezoid(curve, dx=1 / steps)), curve

everything = dict(MAPS, **{'random map': torch.rand(224, 224)})
report = {}
for name, heat in everything.items():
    d, dc = curve_auc(heat, 'deletion')
    i, ic = curve_auc(heat, 'insertion')
    report[name] = (d, i, dc, ic)
    print(f'{name:20} deletion {d:.4f} (lower is better)   insertion {i:.4f} (higher is better)')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
grid = np.linspace(0, 1, 101)
for name, (d, i, dc, ic) in report.items():
    style = dict(lw=2.5, color='crimson') if name == 'random map' else dict(lw=1.2)
    ax1.plot(grid, dc, label=f'{name} ({d:.3f})', **style)
    ax2.plot(grid, ic, label=f'{name} ({i:.3f})', **style)
ax1.set_title('Deletion: the lower the curve, the better the map')
ax2.set_title('Insertion: the higher the curve, the better the map')
for ax in (ax1, ax2):
    ax.set_xlabel('fraction of pixels changed')
    ax.set_ylabel('confidence, as a fraction of the original')
    ax.grid(alpha=.3)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

best_del = min(report, key=lambda k: report[k][0])
best_ins = max(report, key=lambda k: report[k][1])
print(f'best by deletion:   {best_del}')
print(f'best by insertion: {best_ins}')

## 6. The metric depends on what you «remove» with

Both metrics agreed. But they hide the same parameter as the methods themselves — **what the
removed part is replaced with**. We took a blur. Let us take black and recompute deletion,
changing nothing else.

In [ ]:
def deletion_with(heat, filler, steps=100):
    order = heat.flatten().argsort(descending=True)
    per = len(order) // steps
    cur = x.clone()
    curve = [1.0]
    for t in range(steps):
        idx = order[t * per:(t + 1) * per]
        rows, cols = idx // 224, idx % 224
        cur[0, :, rows, cols] = filler[:, rows, cols]
        curve.append(predict(cur).item() / base_score)
    return float(np.trapezoid(curve, dx=1 / steps))

print(f'{"":22}{"blur  ":>12}{"black ":>12}')
for name in ('Occlusion, window 64', 'RISE, 200 masks', 'random map'):
    h = everything[name]
    print(f'{name:22}{deletion_with(h, blur):>12.4f}{deletion_with(h, black):>12.4f}')

With one word changed in the definition of the metric, the random map turns from an
obvious outsider into the winner: it beats both RISE and occlusion — the latter by almost an
order of magnitude.

The reason is not in the maps but in the perturbation itself. A black rectangle in the middle of
a photograph is an object that never occurred in the training set; the random map scatters such
patches evenly and turns the input into salt-and-pepper noise that breaks every texture at once.
The confidence collapses fast, and the metric honestly records that as «a good map».

So deletion measures not only the faithfulness of the map but also the **distribution shift**
introduced by the removal itself. This is exactly the criticism that ROAR and its relatives were
invented for, and exactly the reason why in the block on evaluating explanations we never rely
on a single metric.

**Task 1.** Sort the methods by insertion. Did the order match your impression of the pictures
in section 4?

**Task 2.** Rebuild the maps with `BASELINE = black` inside the methods themselves (not only
inside the metric). How much does occlusion change? And RISE, which does not use a black
baseline at all?

**Task 3.** Check how insertion changes if you remove not single pixels but $8\times8$ blocks
(hint: average the map with `avg_pool2d` before sorting). Does the random map stay the worst?

## 7. What the parameter $p$ does

The lesson says: «a low $p$ zeroes out too many pixels, a high $p$ hardly changes the picture».
Let us look at the spread of the map — if it collapses, the method has stopped telling regions
apart even though it formally ran.

In [ ]:
print(f'{"p":>5}{"insertion":>12}{"spread of map":>16}')
for p in (0.1, 0.3, 0.5, 0.7, 0.9):
    heat = rise(200, p=p)
    ins, _ = curve_auc(heat, 'insertion')
    print(f'{p:>5}{ins:>12.4f}{(heat.max() - heat.min()).item():>16.4f}')

**Task 4.** At which $p$ is the spread largest? Does it coincide with the $p$ at which
insertion is largest? If not — which of the two matters more for the reader of a report?

**Task 5.** Occlusion with a window of 32 needs more passes than with a window of 64. Work out
how many times more, and compare that with the gain in insertion. Was it worth it?

## What to take away

- **All three methods are one formula** $f(x) - f(x^{(S\to b)})$ with a different $S$: a single
  pixel for naive ablation, a sliding window for occlusion, a random mask for RISE.
- **One pixel at a time does not work.** The contribution of a single pixel is thousandths of a
  percent: the network restores it from the neighbours. For images, perturbation is always grouped.
- **The baseline is part of the method, not an implementation detail.** The same grey
  `zeros_like` trap as in Integrated Gradients, and the same choice between black and blur.
- **A faithfulness metric can itself be unfaithful.** Deletion with an aggressive baseline awards
  first place to a random map, because it measures the network's reaction to the artefact of the
  perturbation rather than the importance of the pixels. One metric is not enough — you need two,
  and they have to agree.
- **The price.** Occlusion 32/16 is hundreds of forward passes, RISE 500 is five hundred.
  Gradient methods get by with a single backward pass. Independence from the internals of the
  model is paid for in time.

**Labels:** local, post-hoc, model-agnostic — only the input and the output are needed. Input: a
model as a black box and one image. Output: an importance map the size of the input.